# 01 · Chạy inference 1 model

Chạy **00_setup trước**. Mỗi model chạy trong **1 session riêng** (deps xung đột nhau).
Chọn `MODEL` bên dưới rồi Run all. An toàn khi chạy lại: đã có utt thì bỏ qua (resume).


In [ ]:
%cd /content/benchmark-for-asr-model
MODEL = 'gipformer_65m'  # gipformer_65m | zipformer_30m | chunkformer_large | nemotron_06b
DATASETS = ['auto_autoele', 'sol_ups']
REQ = {'gipformer_65m':'gipformer', 'zipformer_30m':'zipformer',
       'chunkformer_large':'chunkformer', 'nemotron_06b':'nemotron'}[MODEL]

### Cài deps riêng của model
> **nemotron**: bỏ comment dòng apt bên dưới trước khi cài.


In [ ]:
# !apt-get -qq install -y libsndfile1 ffmpeg && pip install -q Cython packaging  # chỉ nemotron
!pip install -q -r requirements/{REQ}.txt

### (Chỉ zipformer) sinh tokens.txt từ bpe.model
Repo zipformer không có sẵn tokens.txt. Chạy cell này rồi trỏ `tokens` trong config tới file sinh ra.


In [ ]:
if MODEL == 'zipformer_30m':
    from huggingface_hub import hf_hub_download
    import sentencepiece as spm, os
    bpe = hf_hub_download('hynt/Zipformer-30M-RNNT-Streaming-6000h', 'bpe.model')
    sp = spm.SentencePieceProcessor(); sp.load(bpe)
    os.makedirs('models', exist_ok=True)
    with open('models/zipformer_tokens.txt','w') as f:
        for i in range(sp.get_piece_size()):
            f.write(f'{sp.id_to_piece(i)} {i}\n')
    print('tokens.txt -> models/zipformer_tokens.txt ; sửa configs/models/zipformer_30m.yaml: tokens: models/zipformer_tokens.txt')

### Smoke-test 5 utterance trước khi chạy full


In [ ]:
!PYTHONPATH=src python -m benchmark.runners.run_inference \
  --model-config configs/models/{MODEL}.yaml \
  --manifest data/manifests/auto_autoele.jsonl \
  --out /tmp/smoke_{MODEL}.jsonl --limit 5 --no-resume

### Chạy full (resume-được; lưu ra Drive)


In [ ]:
for ds in DATASETS:
    print('=== ', MODEL, ds, ' ===')
    !PYTHONPATH=src python -m benchmark.runners.run_inference \
        --model-config configs/models/{MODEL}.yaml \
        --manifest data/manifests/{ds}.jsonl \
        --out results/hypotheses/{MODEL}/{ds}.jsonl --warmup 3